In [1]:
# ============================================================
# CALL CENTER CONVERSATION ANALYSIS SYSTEM
# Single-cell Google Colab Application
# ============================================================

# Install required libraries
!pip -q install transformers sentencepiece torch nltk

import re
import nltk
import torch
from collections import Counter
from transformers import pipeline

nltk.download("punkt", quiet=True)

print("=" * 65)
print("       AI CALL CENTER CONVERSATION ANALYSIS SYSTEM")
print("=" * 65)

# ------------------------------------------------------------
# 1. Load AI Models
# ------------------------------------------------------------

print("\nLoading AI models... Please wait.")

# Sentiment analysis model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Emotion analysis model
emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=1
)

print("AI models loaded successfully!")


# ------------------------------------------------------------
# 2. Get Call Center Conversation
# ------------------------------------------------------------

conversation = input("""
Enter the customer-agent conversation.

Example:
Customer: I have been waiting for my order for five days.
Agent: I am sorry for the delay. Let me check your order.
Customer: I need it urgently. Nobody has updated me.
Agent: I understand your concern. Your order will be delivered tomorrow.
Customer: Thank you for helping me.

Your conversation:
""")


# ------------------------------------------------------------
# 3. Basic Text Cleaning
# ------------------------------------------------------------

clean_text = re.sub(r'\s+', ' ', conversation).strip()

if len(clean_text) < 10:
    print("\nPlease enter a longer conversation.")
    raise SystemExit

print("\n" + "=" * 65)
print("                 ANALYSIS STARTED")
print("=" * 65)


# ------------------------------------------------------------
# 4. Separate Customer and Agent Messages
# ------------------------------------------------------------

customer_messages = re.findall(
    r'Customer\s*:\s*(.*?)(?=Agent\s*:|$)',
    conversation,
    re.IGNORECASE
)

agent_messages = re.findall(
    r'Agent\s*:\s*(.*?)(?=Customer\s*:|$)',
    conversation,
    re.IGNORECASE
)

customer_text = " ".join(customer_messages)
agent_text = " ".join(agent_messages)


# ------------------------------------------------------------
# 5. Overall Sentiment Analysis
# ------------------------------------------------------------

sentiment_result = sentiment_model(clean_text[:512])[0]

sentiment = sentiment_result["label"]
confidence = sentiment_result["score"] * 100

if sentiment == "POSITIVE":
    sentiment_display = "Positive 😊"
else:
    sentiment_display = "Negative 😟"


# ------------------------------------------------------------
# 6. Emotion Analysis
# ------------------------------------------------------------

emotion_result = emotion_model(clean_text[:512])[0][0]

emotion = emotion_result["label"]
emotion_confidence = emotion_result["score"] * 100

emotion_names = {
    "anger": "Angry 😠",
    "disgust": "Disgusted 🤢",
    "fear": "Fearful 😨",
    "joy": "Happy 😊",
    "neutral": "Neutral 😐",
    "sadness": "Sad 😢",
    "surprise": "Surprised 😮"
}

emotion_display = emotion_names.get(
    emotion.lower(),
    emotion.capitalize()
)


# ------------------------------------------------------------
# 7. Customer Sentiment
# ------------------------------------------------------------

if customer_text:
    customer_sentiment = sentiment_model(
        customer_text[:512]
    )[0]

    customer_sentiment_label = customer_sentiment["label"]

    if customer_sentiment_label == "POSITIVE":
        customer_sentiment_display = "Positive 😊"
    else:
        customer_sentiment_display = "Negative 😟"
else:
    customer_sentiment_display = "Not detected"


# ------------------------------------------------------------
# 8. Detect Customer Issues
# ------------------------------------------------------------

issue_keywords = {
    "Delivery Problem": [
        "delivery", "delivered", "shipping", "shipment",
        "late", "delay", "order", "package"
    ],

    "Payment Problem": [
        "payment", "paid", "money", "refund",
        "transaction", "card", "billing", "charge"
    ],

    "Technical Problem": [
        "error", "bug", "not working", "crash",
        "problem", "issue", "login", "password"
    ],

    "Product Problem": [
        "damaged", "broken", "defective",
        "wrong product", "replacement"
    ],

    "Account Problem": [
        "account", "username", "password",
        "profile", "verification"
    ],

    "Cancellation Request": [
        "cancel", "cancellation", "stop my order"
    ]
}

detected_issues = []

lower_text = clean_text.lower()

for issue, keywords in issue_keywords.items():
    for keyword in keywords:
        if keyword in lower_text:
            detected_issues.append(issue)
            break

if not detected_issues:
    detected_issues.append("General Inquiry")


# ------------------------------------------------------------
# 9. Extract Important Keywords
# ------------------------------------------------------------

stop_words = {
    "the", "is", "a", "an", "and", "or", "to", "of",
    "in", "on", "for", "with", "this", "that", "it",
    "i", "you", "we", "my", "your", "me", "are",
    "was", "were", "be", "have", "has", "had",
    "can", "will", "would", "could", "please",
    "am", "do", "does", "did", "but", "so",
    "very", "not", "from", "our", "they", "he",
    "she", "them", "their"
}

words = re.findall(r'\b[a-zA-Z]{3,}\b', lower_text)

filtered_words = [
    word for word in words
    if word not in stop_words
]

word_frequency = Counter(filtered_words)

top_keywords = word_frequency.most_common(8)


# ------------------------------------------------------------
# 10. Detect Urgency
# ------------------------------------------------------------

urgency_words = [
    "urgent",
    "urgently",
    "immediately",
    "asap",
    "emergency",
    "quickly",
    "soon",
    "important"
]

urgency_detected = any(
    word in lower_text for word in urgency_words
)

if urgency_detected:
    urgency = "HIGH 🔴"
else:
    urgency = "NORMAL 🟢"


# ------------------------------------------------------------
# 11. Detect Escalation
# ------------------------------------------------------------

escalation_words = [
    "manager",
    "supervisor",
    "complaint",
    "legal",
    "court",
    "escalate",
    "escalation"
]

escalation_detected = any(
    word in lower_text for word in escalation_words
)

if escalation_detected:
    escalation = "Required ⚠️"
else:
    escalation = "Not Required ✅"


# ------------------------------------------------------------
# 12. Agent Performance Analysis
# ------------------------------------------------------------

agent_positive_words = [
    "sorry",
    "understand",
    "help",
    "thank",
    "please",
    "assist",
    "certainly",
    "sure",
    "resolve"
]

agent_lower = agent_text.lower()

agent_helpfulness = sum(
    1 for word in agent_positive_words
    if word in agent_lower
)

if agent_helpfulness >= 3:
    agent_rating = "Excellent ⭐⭐⭐⭐⭐"
elif agent_helpfulness >= 2:
    agent_rating = "Good ⭐⭐⭐⭐"
elif agent_helpfulness >= 1:
    agent_rating = "Average ⭐⭐⭐"
else:
    agent_rating = "Needs Improvement ⭐⭐"


# ------------------------------------------------------------
# 13. Customer Satisfaction Score
# ------------------------------------------------------------

satisfaction_score = 50

if sentiment == "POSITIVE":
    satisfaction_score += 25
else:
    satisfaction_score -= 20

if customer_sentiment_display == "Positive 😊":
    satisfaction_score += 20
elif customer_sentiment_display == "Negative 😟":
    satisfaction_score -= 15

if urgency_detected:
    satisfaction_score -= 5

if escalation_detected:
    satisfaction_score -= 10

satisfaction_score = max(
    0,
    min(100, satisfaction_score)
)

if satisfaction_score >= 80:
    satisfaction_level = "Highly Satisfied 😊"
elif satisfaction_score >= 60:
    satisfaction_level = "Satisfied 🙂"
elif satisfaction_score >= 40:
    satisfaction_level = "Neutral 😐"
else:
    satisfaction_level = "Dissatisfied 😟"


# ------------------------------------------------------------
# 14. Generate Automatic Summary
# ------------------------------------------------------------

if detected_issues[0] == "Delivery Problem":
    summary = "The customer appears to have a delivery or order-related concern."

elif detected_issues[0] == "Payment Problem":
    summary = "The customer appears to have a payment or billing-related concern."

elif detected_issues[0] == "Technical Problem":
    summary = "The customer appears to have a technical or application-related issue."

elif detected_issues[0] == "Product Problem":
    summary = "The customer appears to have a problem with the product."

elif detected_issues[0] == "Account Problem":
    summary = "The customer appears to have an account-related issue."

elif detected_issues[0] == "Cancellation Request":
    summary = "The customer appears to be requesting cancellation."

else:
    summary = "The conversation appears to be a general customer inquiry."


# ------------------------------------------------------------
# 15. Display Final Report
# ------------------------------------------------------------

print("\n")
print("=" * 65)
print("                  ANALYSIS REPORT")
print("=" * 65)

print("\n📌 CONVERSATION DETAILS")
print("-" * 65)

print("Customer Messages :", len(customer_messages))
print("Agent Messages    :", len(agent_messages))
print("Total Words       :", len(words))


print("\n😊 SENTIMENT ANALYSIS")
print("-" * 65)

print("Overall Sentiment :", sentiment_display)
print("Confidence        :", f"{confidence:.2f}%")
print("Customer Sentiment:", customer_sentiment_display)


print("\n🎭 EMOTION ANALYSIS")
print("-" * 65)

print("Detected Emotion  :", emotion_display)
print("Confidence        :", f"{emotion_confidence:.2f}%")


print("\n🔍 ISSUE DETECTION")
print("-" * 65)

for issue in detected_issues:
    print("•", issue)


print("\n🚨 PRIORITY ANALYSIS")
print("-" * 65)

print("Urgency           :", urgency)
print("Escalation        :", escalation)


print("\n👨‍💼 AGENT PERFORMANCE")
print("-" * 65)

print("Agent Rating      :", agent_rating)


print("\n🔑 IMPORTANT KEYWORDS")
print("-" * 65)

if top_keywords:
    for keyword, count in top_keywords:
        print(f"• {keyword} ({count})")
else:
    print("No important keywords detected.")


print("\n📊 CUSTOMER SATISFACTION")
print("-" * 65)

print("Satisfaction Score:", f"{satisfaction_score}/100")
print("Satisfaction Level:", satisfaction_level)


print("\n📝 AUTOMATIC SUMMARY")
print("-" * 65)

print(summary)


print("\n💡 RECOMMENDATION")
print("-" * 65)

if escalation_detected:
    print("Escalate this conversation to a supervisor or senior support agent.")

elif urgency_detected:
    print("Prioritize this customer because the conversation indicates urgency.")

elif sentiment == "NEGATIVE":
    print("The agent should follow up with the customer and resolve the issue quickly.")

else:
    print("The conversation appears to be handled reasonably well.")


print("\n" + "=" * 65)
print("              ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 65)

       AI CALL CENTER CONVERSATION ANALYSIS SYSTEM

Loading AI models... Please wait.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

AI models loaded successfully!

Enter the customer-agent conversation.

Example:
Customer: I have been waiting for my order for five days.
Agent: I am sorry for the delay. Let me check your order.
Customer: I need it urgently. Nobody has updated me.
Agent: I understand your concern. Your order will be delivered tomorrow.
Customer: Thank you for helping me.

Your conversation:
hi i am

Please enter a longer conversation.


SystemExit: 

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
